In [3]:
import nest_asyncio
nest_asyncio.apply()

In [6]:
from playwright.async_api import async_playwright
import pandas as pd
import re

def clean_df(df):

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df["old_price"] = pd.to_numeric(df["old_price"], errors="coerce")

    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    df["review_count"] = pd.to_numeric(
        df["review_count"],
        errors="coerce"
    )

    df["sales_last_month"] = pd.to_numeric(
        df["sales_last_month"],
        errors="coerce"
    )

    for col in ["title", "link", "asin"]:
        df[col] = df[col].astype(str)

    df = df.drop_duplicates(subset=["asin"])

    return df


def convert_price(price_text):

    try:

        price_text = price_text.replace(".", "")
        price_text = price_text.replace(",", ".")

        return float(price_text)

    except:
        return None


async def scrape_amazon():

    all_data = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=False
        )

        page = await browser.new_page()

        # 40 SAYFA
        for page_num in range(1, 61):

            url = (
                f"https://www.amazon.com.tr/s?"
                f"k=kozmetik&page={page_num}"
            )

            print(f"\nScraping page {page_num}")

            try:

                await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=120000
                )

                await page.wait_for_timeout(4000)

            except Exception as e:

                print("Sayfa yüklenemedi:", e)
                continue

            items = await page.query_selector_all(
                "[data-component-type='s-search-result']"
            )

            print("Bulunan ürün:", len(items))

            for item in items:

                asin = await item.get_attribute("data-asin")

                if not asin:
                    continue

                try:

                
                    title_el = await item.query_selector(
                        "h2 span"
                    )

                    title = (
                        await title_el.inner_text()
                        if title_el
                        else None
                    )

                  
                    text_content = await item.inner_text()

                    prices = re.findall(
                        r"(\d{1,3}(?:\.\d{3})*,\d{2})\s*TL",
                        text_content
                    )

                    price = None
                    old_price = None

                    if len(prices) == 1:

                        price = convert_price(
                            prices[0]
                        )

                    elif len(prices) >= 2:

                        old_price = convert_price(
                            prices[0]
                        )

                        price = convert_price(
                            prices[1]
                        )

                    rating = None

                    rating_el = await item.query_selector(
                        ".a-icon-alt"
                    )

                    if rating_el:

                        rating_text = (
                            await rating_el.inner_text()
                        )

                        match = re.search(
                            r"(\d+[,.]\d)",
                            rating_text
                        )

                        if match:

                            rating = float(
                                match.group(1)
                                .replace(",", ".")
                            )


                    review_count = None

                    try:

                        review_link = await item.query_selector(
                            "a[href*='customerReviews']"
                        )

                        if review_link:

                            review_text = (
                                await review_link.inner_text()
                            )

                            review_match = re.search(
                                r"([\d\.]+)",
                                review_text
                            )

                            if review_match:

                                review_count = int(
                                    review_match.group(1)
                                    .replace(".", "")
                                )

                    except:
                        pass

                    if review_count is None:

                        review_match = re.search(
                            r"([\d\.]+)\s*değerlendirme",
                            text_content,
                            re.IGNORECASE
                        )

                        if review_match:

                            review_count = int(
                                review_match.group(1)
                                .replace(".", "")
                            )

            

                    sales_last_month = None

                    sales_match = re.search(
                        r"Geçen ay\s+(\d+)",
                        text_content,
                        re.IGNORECASE
                    )

                    if sales_match:

                        sales_last_month = int(
                            sales_match.group(1)
                        )

            
                    link = None

                    link_el = await item.query_selector(
                        "h2 a"
                    )

                    if link_el:

                        href = await link_el.get_attribute(
                            "href"
                        )

                        if href:

                            link = (
                                "https://www.amazon.com.tr"
                                + href
                            )

                    all_data.append({

                        "asin": asin,
                        "title": title,
                        "price": price,
                        "old_price": old_price,
                        "rating": rating,
                        "review_count": review_count,
                        "sales_last_month": sales_last_month,
                        "link": link

                    })

                except Exception as e:

                    print(
                        "Ürün hatası:",
                        e
                    )

        await browser.close()

    df = pd.DataFrame(all_data)

    df = clean_df(df)

    print("Toplam kayıt:", len(df))

    df.to_csv(
        "amazon_kozmetik_final.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "CSV kaydedildi -> amazon_kozmetik_final_3.csv"
    )

    print(df.head())


await scrape_amazon()


Scraping page 1
Bulunan ürün: 60

Scraping page 2
Bulunan ürün: 60

Scraping page 3
Bulunan ürün: 60

Scraping page 4
Bulunan ürün: 60

Scraping page 5
Bulunan ürün: 60

Scraping page 6
Bulunan ürün: 60

Scraping page 7
Bulunan ürün: 30

Scraping page 8
Bulunan ürün: 12

Scraping page 9
Bulunan ürün: 12

Scraping page 10
Bulunan ürün: 12

Scraping page 11
Bulunan ürün: 12

Scraping page 12
Bulunan ürün: 12

Scraping page 13
Bulunan ürün: 12

Scraping page 14
Bulunan ürün: 12

Scraping page 15
Bulunan ürün: 12

Scraping page 16
Bulunan ürün: 12

Scraping page 17
Bulunan ürün: 12

Scraping page 18
Bulunan ürün: 12

Scraping page 19
Bulunan ürün: 12

Scraping page 20
Bulunan ürün: 12

Scraping page 21
Bulunan ürün: 12

Scraping page 22
Bulunan ürün: 12

Scraping page 23
Bulunan ürün: 12

Scraping page 24
Bulunan ürün: 12

Scraping page 25
Bulunan ürün: 12

Scraping page 26
Bulunan ürün: 12

Scraping page 27
Bulunan ürün: 12

Scraping page 28
Bulunan ürün: 12

Scraping page 29
Bulunan ürü